# Query Decomposition / Multi-Query Retrieval

**The problem:** one user question is often a *single* point in embedding space, and similarity
search only looks near that point. A broad or multi-part question ("How does a student get admitted
to VIT and what entrance exams are involved?") has several facets, each living in a different part
of the document.

**Query decomposition:** ask the LLM to break the question into several focused **sub-questions**,
run similarity search for **each** (plus the original), then answer from the **de-duplicated union**
of everything retrieved. More search points -> better coverage of a many-faceted question.

LangChain's `MultiQueryRetriever` does exactly this: prompt -> LLM -> split into sub-questions ->
retrieve each -> merge.

**Document:** `small.pdf` - VIT University's FFCS (Fully Flexible Credit System) Academic
Regulations 4.0 (abbreviations, FFCS features, version history, admissions: VITEEE / VITMEE /
V-SIGN). Niche, institution-specific content the base LLM does not know.

**Runs on Google Colab** (or Kaggle) - CPU is fine.

## 1. Install

`MultiQueryRetriever` lives in the **`langchain`** package. `pip install -U langchain`
now pulls **LangChain 1.0**, which restructured those modules, so we pin the whole
family to the **0.3** line.

In [ ]:
!pip install -q "langchain>=0.3,<1.0" "langchain-core>=0.3,<1.0" "langchain-community>=0.3,<1.0" "langchain-text-splitters>=0.3,<1.0" "langchain-huggingface>=0.1,<1.0" "langchain-chroma>=0.1,<1.0" "langchain-google-genai>=2.0,<3.0"
!pip install -q sentence-transformers chromadb pypdf

> ### Restart the runtime now
> **Runtime -> Restart session**, then run every cell from the top. Without a restart the
> freshly pinned `langchain` may not be picked up.

Sanity check after the restart: the `MultiQueryRetriever` import proves the 0.3 pin took effect.

In [ ]:
# verify (run AFTER restarting)
import langchain
from langchain.retrievers.multi_query import MultiQueryRetriever
print("langchain", langchain.__version__, "- MultiQueryRetriever import OK")

## 2. Imports

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain.retrievers.multi_query import MultiQueryRetriever
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel, RunnablePassthrough

## 3. The LLM (Google Gemini)

In [ ]:
import os
import warnings

# chromadb telemetry off (avoids opentelemetry noise/errors)
os.environ["ANONYMIZED_TELEMETRY"] = "False"
os.environ["CHROMA_TELEMETRY_ENABLED"] = "False"

# gemini-3.5-flash-lite uses fixed sampling -> silence the "temperature ignored" notice
warnings.filterwarnings("ignore", message=".*fixed sampling defaults.*")

# your Gemini key: https://aistudio.google.com/apikey
os.environ["GOOGLE_API_KEY"] = "YOUR_GOOGLE_API_KEY"

from langchain_google_genai import ChatGoogleGenerativeAI
llm = ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite")

Paste your Gemini key (or load from Secrets). This one LLM does **two** jobs in this notebook:
generate the sub-questions, and write the final grounded answer.

## 4. Load, split, and index the PDF

In [ ]:
# small local embedding model (CPU is fine)
embedding_function = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

`all-MiniLM-L6-v2` - small, fast, local. Embeds the document chunks.

In [ ]:
# --- small.pdf : VIT FFCS academic regulations write-up ---
# Colab : upload it via the Files panel           -> "/content/small.pdf"
# Kaggle: Add Input -> your dataset with the file -> "/kaggle/input/<slug>/small.pdf"
PDF_PATH = "/content/small.pdf"

import os
assert os.path.exists(PDF_PATH), f"PDF not found at {PDF_PATH!r} - upload small.pdf and fix PDF_PATH"

loader = PyPDFLoader(PDF_PATH)
documents = loader.load()
print("loaded", len(documents), "pages")

Set `PDF_PATH` for your environment. `documents` = one `Document` per page.

In [ ]:
# small.pdf is short; a smallish chunk gives the retriever more to work with
# (the original notebook's chunk_size=100 was far too small)
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
texts = text_splitter.split_documents(documents)
print(len(texts), "chunks")

vectorstore = Chroma(
    collection_name="ffcs_multiquery",
    embedding_function=embedding_function,
)
vectorstore.add_documents(texts)
print("indexed", len(vectorstore.get()["ids"]), "chunks")

Split into 500-char chunks (overlap 100) and index them in one in-memory `Chroma` collection.
The `k=3`-per-query setting is applied on the retriever in the next section.

## 5. The MultiQueryRetriever

`MultiQueryRetriever.from_llm` wires up: **prompt -> LLM -> split into lines**. Each line
is a sub-question; the retriever runs similarity search for every sub-question (and,
with `include_original=True`, the original too) and returns the **de-duplicated union**.

We pass a custom prompt so you can shape how the question is decomposed.

In [ ]:
decompose_prompt = PromptTemplate(
    input_variables=["question"],
    template=(
        "You are helping search a document about VIT University's FFCS academic regulations.\n"
        "Break the user's question into 4 focused sub-questions that together fully cover it.\n"
        "Return ONLY the sub-questions, one per line, with no numbering or extra text.\n\n"
        "Question: {question}"
    ),
)

multiquery_retriever = MultiQueryRetriever.from_llm(
    retriever=vectorstore.as_retriever(search_kwargs={"k": 3}),
    llm=llm,
    prompt=decompose_prompt,
    include_original=True,
)

`decompose_prompt` tells the LLM *how* to split the question (here: exactly 4 sub-questions,
one per line). `MultiQueryRetriever.from_llm` wires prompt -> LLM -> split-into-lines -> retrieve
each; `include_original=True` keeps the user's original question in the search set too.

In [ ]:
# turn on INFO logging so you can SEE the generated sub-questions
import logging
logging.basicConfig()
logging.getLogger("langchain.retrievers.multi_query").setLevel(logging.INFO)

question = "How does a student get admitted to VIT and what entrance exams are involved?"
docs = multiquery_retriever.invoke(question)

print(f"\n{len(docs)} unique chunks retrieved across all sub-questions. First 3:\n")
for d in docs[:3]:
    print("-", d.page_content[:200].replace("\n", " "), "...\n")

Turning on INFO logging prints the **generated sub-questions**, so you can see the
decomposition. The returned `docs` list is the **de-duplicated union** of the chunks retrieved
across every sub-question.

## 6. RAG chain on top of the multi-query retriever

In [ ]:
def format_docs(docs):
    return "\n\n---\n\n".join(d.page_content for d in docs)

gen_prompt = ChatPromptTemplate.from_template(
    "You are a precise assistant answering questions about VIT University's FFCS academic\n"
    "regulations document. Answer ONLY from the context below. If it is not in the context, say so.\n\n"
    "Context:\n{context}\n\n"
    "Question: {question}\n"
)

query_decomposition_chain = (
    RunnableParallel(
        {"context": multiquery_retriever | format_docs, "question": RunnablePassthrough()}
    )
    | gen_prompt
    | llm
    | StrOutputParser()
)

`RunnableParallel` builds `{context: multiquery_retriever | format_docs, question: passthrough}`,
then `gen_prompt | llm | StrOutputParser`. The prompt restricts the model to the retrieved context.

In [ ]:
print(query_decomposition_chain.invoke(
    "What is FFCS and what are its main features?"
))

## 7. Try several questions

In [ ]:
questions = [
    "What do the abbreviations DLE, DC, FC and CAL stand for?",
    "What entrance exams does VIT conduct, and for which programmes?",
    "What is V-SIGN, when did it start, and what programmes does it offer?",
    "What role does a proctor or faculty advisor play under FFCS?",
]
for q in questions:
    print("=" * 90)
    print("Q:", q)
    print("-" * 90)
    print(query_decomposition_chain.invoke(q))
    print()

Watch the logged sub-questions for each: broad questions get split into several narrower searches.

## 8. Optional: interactive chat loop

`input()` works in Colab and Kaggle notebooks. Type **STOP** to exit.

In [ ]:
import warnings, logging
warnings.filterwarnings("ignore")

# quieten the per-query "Generated queries" logging for the chat
logging.getLogger("langchain.retrievers.multi_query").setLevel(logging.WARNING)

def query_decomp_chat():
    print("Ask me about VIT's FFCS academic regulations. Type STOP to exit.")
    print("-" * 80)
    question = input()
    while question.strip().upper() != "STOP":
        print(query_decomposition_chain.invoke(question))   # pass the question string directly
        print("\nAnything else?")
        print("-" * 80)
        question = input()

# query_decomp_chat()   # <- uncomment to run